# Project: Internal IT Helpdesk Agent
**Brief from Paras:**
Monitors Slack/email for IT requests, classifies (access/reset/install),
searches Notion runbooks, auto-resolves simple ones or creates Linear ticket, notifies user.

This is a SCAFFOLD. The supervisor + graph wiring is complete; worker logic for some nodes is marked TODO.
Use Project #2 (`02_support_resolver/support_resolver.ipynb`) as your reference implementation.

## Submission checklist
- [ ] All worker TODOs filled in
- [ ] Composio actions verified against docs.composio.dev
- [ ] HITL where destructive actions occur
- [ ] Checkpoint persistence configured
- [ ] Graph diagram saved as graph.png
- [ ] README.md with architecture + example run


## 1. Setup

In [ ]:
import os, sqlite3
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(".env")

assert os.getenv("OPENAI_API_KEY")
assert os.getenv("COMPOSIO_API_KEY"), "Sign up at composio.dev and connect required apps in their dashboard"
print("env OK")


## 2. State schema

In [ ]:
from typing import TypedDict, Annotated, Optional, Literal
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage, SystemMessage

class ItHelpdeskState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    next_worker: str
    user_id: str
    request_text: str
    category: Optional[str]
    runbook_match: Optional[str]
    linear_ticket_id: Optional[str]
    auto_resolved: bool


## 3. LLM and Composio toolset

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from composio_langgraph import Action, ComposioToolSet

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
toolset = ComposioToolSet()


## 4. Workers

In [ ]:
# classifier - Request classifier (pure LLM)
# No Composio actions; pure LLM or custom logic.
# TODO: implement classifier_node(state) following the pattern in 02_support_resolver.
def classifier_node(state):
    """Classify into one of: access (VPN, app access), reset (password, 2FA), install (software request), other."""
    raise NotImplementedError('TODO: implement classifier_node')


# runbook_finder - Notion runbook finder
runbook_finder_tools = toolset.get_tools(actions=[Action.NOTION_SEARCH_NOTION_PAGE])
runbook_finder_agent = create_react_agent(llm, runbook_finder_tools, prompt='''Search Notion 'Runbooks' for matching runbook. Return runbook content.''')
def runbook_finder_node(state):
    result = runbook_finder_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='runbook_finder')]}


# auto_resolver - Self-service resolver
auto_resolver_tools = toolset.get_tools(actions=[Action.SLACK_SENDS_A_MESSAGE_TO_A_SLACK_CHANNEL])
auto_resolver_agent = create_react_agent(llm, auto_resolver_tools, prompt='''If runbook covers self-service (e.g. password reset link), DM the user the steps and mark resolved.''')
def auto_resolver_node(state):
    result = auto_resolver_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='auto_resolver')]}


# ticket_escalator - Linear ticket creator
ticket_escalator_tools = toolset.get_tools(actions=[Action.LINEAR_CREATE_LINEAR_ISSUE])
ticket_escalator_agent = create_react_agent(llm, ticket_escalator_tools, prompt='''If not self-resolvable, create Linear ticket with category label and assignee.''')
def ticket_escalator_node(state):
    result = ticket_escalator_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='ticket_escalator')]}


# status_notifier - User status notifier
status_notifier_tools = toolset.get_tools(actions=[Action.SLACK_SENDS_A_MESSAGE_TO_A_SLACK_CHANNEL])
status_notifier_agent = create_react_agent(llm, status_notifier_tools, prompt='''Send DM to user with resolution or ticket link.''')
def status_notifier_node(state):
    result = status_notifier_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='status_notifier')]}


## 5. Supervisor + router

In [ ]:
def supervisor(state) -> dict:
    if state.get('category') is None: return {'next_worker': 'classifier'}
    if state.get('runbook_match') is None and 'runbook_searched' not in str(state['messages']):
        return {'next_worker': 'runbook_finder'}
    if state['runbook_match'] and not state['auto_resolved']:
        return {'next_worker': 'auto_resolver'}
    if not state['auto_resolved'] and state.get('linear_ticket_id') is None:
        return {'next_worker': 'ticket_escalator'}
    if 'user notified' not in (state['messages'][-1].content.lower() if state['messages'] else ''):
        return {'next_worker': 'status_notifier'}
    return {'next_worker': 'DONE'}

def route(state) -> str:
    nxt = state['next_worker']
    if nxt in {'classifier', 'auto_resolver', 'runbook_finder', 'ticket_escalator', 'status_notifier'}:
        return nxt
    return '__end__'


## 6. Build graph + checkpoint persistence

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver

g = StateGraph(globals()[[k for k in globals() if k.endswith('State') and k != 'AgentState'][0]])
g.add_node("supervisor", supervisor)
g.add_node("classifier", classifier_node)
g.add_node("runbook_finder", runbook_finder_node)
g.add_node("auto_resolver", auto_resolver_node)
g.add_node("ticket_escalator", ticket_escalator_node)
g.add_node("status_notifier", status_notifier_node)

g.add_edge(START, "supervisor")
g.add_conditional_edges("supervisor", route, {
    "classifier": "classifier",
    "runbook_finder": "runbook_finder",
    "auto_resolver": "auto_resolver",
    "ticket_escalator": "ticket_escalator",
    "status_notifier": "status_notifier",
    "__end__": END,
})
g.add_edge("classifier", "supervisor")
g.add_edge("runbook_finder", "supervisor")
g.add_edge("auto_resolver", "supervisor")
g.add_edge("ticket_escalator", "supervisor")
g.add_edge("status_notifier", "supervisor")

conn = sqlite3.connect("10_it_helpdesk.db", check_same_thread=False)
app = g.compile(checkpointer=SqliteSaver(conn))


## 7. Visualise (submission requirement)

In [ ]:
from IPython.display import Image, display
try:
    png = app.get_graph().draw_mermaid_png()
    Path("graph.png").write_bytes(png)
    display(Image("graph.png"))
except Exception as e:
    print("ASCII fallback:")
    print(app.get_graph().draw_ascii())


## 8. End-to-end demo

In [ ]:
config = {'configurable': {'thread_id': '10_it_helpdesk-demo-1'}, 'recursion_limit': 30}

result = app.invoke(
    {'user_id': 'alice@corp.com',
        'request_text': 'Need to reset my VPN password',
        'messages': [HumanMessage(content='New IT request from Alice')]},
    config=config,
)

print("=== FINAL STATE ===")
for k, v in result.items():
    if k != 'messages':
        print(f"{k}: {str(v)[:150]}")
print("\n=== MESSAGE TRACE ===")
for m in result['messages']:
    label = getattr(m, 'name', m.type)
    print(f"[{label}] {m.content[:200]}")
